# 02 - Feature Engineering (leakage-safe version)

Тук train/test split-ът се прави **преди** да смятаме каквато и да е агрегатна статистика (средни стойности, медиани), за да не изтича информация от test set-а в train features.

In [ ]:
import pandas as pd
import numpy as np

from datetime import datetime
from sklearn.model_selection import train_test_split

pd.set_option("display.max_columns", None)


## Зареждане на данните

In [ ]:
df = pd.read_csv("../data/raw/HRDataset_v14.csv")
df.head()


## Target: Attrition

Провери първо какви уникални стойности има `EmploymentStatus`, за да си сигурна, че списъкът `TERMINATED_STATUSES` покрива правилните категории (напр. да не включва "Leave of Absence").

In [ ]:
df["EmploymentStatus"].unique()


In [ ]:
TERMINATED_STATUSES = ["Voluntarily Terminated", "Terminated for Cause"]

df["Attrition"] = np.where(df["EmploymentStatus"].isin(TERMINATED_STATUSES), 1, 0)
df["Attrition"].value_counts(normalize=True)


## Дати и tenure (безопасно — зависи само от собствения ред)

In [ ]:
date_cols = [
    "DateofHire",
    "DateofTermination",
    "LastPerformanceReview_Date"
]

for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors="coerce")


In [ ]:
# Reference date
today = pd.to_datetime("today")

# Create effective end date:
# - DateofTermination for terminated employees
# - today for active employees
df["EndDate"] = df["DateofTermination"].fillna(today)

# Tenure in years
df["TenureYears"] = (
    (df["EndDate"] - df["DateofHire"])
    .dt.days / 365
)


In [ ]:
df["TenureGroup"] = pd.cut(
    df["TenureYears"],
    bins=[0, 1, 3, 5, 10, 40],
    labels=["<1 year", "1\u20133 years", "3\u20135 years", "5\u201310 years", "10+ years"]
)


## Train / Test split

Тук е ключовата разлика спрямо оригиналния notebook: split-ът се прави **преди** DeptAvgSalary, HighAbsenceFlag и т.н., за да не тече информация от test в train.

In [ ]:
train_df, test_df = train_test_split(
    df, test_size=0.25, random_state=42, stratify=df["Attrition"]
)

train_df = train_df.copy()
test_df = test_df.copy()

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)


## Relative Salary (fit само на train)

In [ ]:
dept_avg_salary = train_df.groupby("Department")["Salary"].mean()

train_df["DeptAvgSalary"] = train_df["Department"].map(dept_avg_salary)
test_df["DeptAvgSalary"] = test_df["Department"].map(dept_avg_salary)

train_df["RelativeSalary"] = train_df["Salary"] / train_df["DeptAvgSalary"]
test_df["RelativeSalary"] = test_df["Salary"] / test_df["DeptAvgSalary"]


## Engagement / Satisfaction флагове

Праговете тук са фиксирани числа (3.5, 3), не статистика от данните — безопасно е да се приложат директно и към двата dataframe-а.

In [ ]:
for d in [train_df, test_df]:
    d["LowEngagementFlag"] = np.where(d["EngagementSurvey"] < 3.5, 1, 0)
    d["LowSatisfactionFlag"] = np.where(d["EmpSatisfaction"] <= 3, 1, 0)


## Absence / Late флагове (fit само на train)

`absence_threshold` (медианата) се учи само от train и после се прилага и към test.

In [ ]:
absence_threshold = train_df["Absences"].median()

train_df["HighAbsenceFlag"] = np.where(train_df["Absences"] > absence_threshold, 1, 0)
test_df["HighAbsenceFlag"] = np.where(test_df["Absences"] > absence_threshold, 1, 0)

train_df["LateRecentlyFlag"] = np.where(train_df["DaysLateLast30"] > 0, 1, 0)
test_df["LateRecentlyFlag"] = np.where(test_df["DaysLateLast30"] > 0, 1, 0)


## Performance score (фиксиран речник — безопасно за двата dataframe-а)

In [ ]:
performance_map = {
    "PIP": 1,
    "Needs Improvement": 2,
    "Fully Meets": 3,
    "Exceeds": 4
}

missing = df["PerformanceScore"][~df["PerformanceScore"].isin(performance_map)].unique()
if len(missing):
    print("\u0412\u043d\u0438\u043c\u0430\u043d\u0438\u0435: \u043d\u0435\u043f\u043e\u0437\u043d\u0430\u0442\u0438 PerformanceScore \u0441\u0442\u043e\u0439\u043d\u043e\u0441\u0442\u0438:", missing)

for d in [train_df, test_df]:
    d["PerformanceScoreNum"] = d["PerformanceScore"].map(performance_map)


## Избор на финални features

In [ ]:
model_features = [
    "TenureYears",
    "RelativeSalary",
    "EngagementSurvey",
    "EmpSatisfaction",
    "SpecialProjectsCount",
    "DaysLateLast30",
    "Absences",
    "PerformanceScoreNum",
    "LowEngagementFlag",
    "LowSatisfactionFlag",
    "HighAbsenceFlag",
    "LateRecentlyFlag"
]

X_train = train_df[model_features]
y_train = train_df["Attrition"]
X_test = test_df[model_features]
y_test = test_df["Attrition"]


## Запис на processed данните

Записваме **два отделни файла** (train / test), вместо един общ `hr_model_ready.csv` — за да не се налага следващият notebook сам да прави split върху вече "изтекли" features.

In [ ]:
train_model = pd.concat([X_train, y_train], axis=1)
test_model = pd.concat([X_test, y_test], axis=1)

train_model.to_csv("../data/processed/hr_model_ready_train.csv", index=False)
test_model.to_csv("../data/processed/hr_model_ready_test.csv", index=False)

print("Train:", train_model.shape)
print("Test:", test_model.shape)
